# Classifier AND/OR

This notebook trains a neural network to simultaneously predict the results of AND and OR logic operations for 2-bit binary inputs.

# 1. Import model from ONNX

In [6]:
import onnxruntime as ort
import numpy as np
import time
from IPython.display import display, Markdown

device = "GPU"
DEVICE_LIST = ["CPU", "GPU"]
if device not in DEVICE_LIST:
    raise ValueError(f"Invalid device: {device}. Must be one of {DEVICE_LIST}")

if device == "GPU":
    providers = ['CUDAExecutionProvider']
else:
    providers = ['CPUExecutionProvider']

try:
    session = ort.InferenceSession("logic_ops_params.onnx", providers=providers)
except Exception as e:
    print(f"ERROR: Failed to initialize {device} session.")
    raise e

input_name = session.get_inputs()[0].name

c:\Users\juanma\0Projs\kv260_ai_projects\venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


# 2. Run tests in batches

In [7]:
# 1. Generate dataset pool purely in NumPy
dataset_size = 100000
# np.random.randint(low, high) is exclusive of high, so 0 to 2 yields 0s and 1s
X_test_batch = np.random.randint(0, 2, (dataset_size, 2)).astype(np.float32)

# Calculate expected outputs for AND, OR, and XOR
y_and_batch = (X_test_batch[:, 0] * X_test_batch[:, 1]).astype(np.int32)
y_or_batch = ((X_test_batch[:, 0] + X_test_batch[:, 1]) > 0).astype(np.int32)
y_xor_batch = (X_test_batch[:, 0] != X_test_batch[:, 1]).astype(np.int32) # Added XOR

# Stack all three targets
y_expected_batch = np.stack([y_and_batch, y_or_batch, y_xor_batch], axis=1)
 
# 2. Test different batch sizes, running exactly 20k iterations for each
batch_sizes = [1, 8, 64, 512, 2048, 16384]
num_iterations = 20000
batch_results = []
 
for batch_size in batch_sizes:
    
    print(f">>> Starting benchmark for batch size: {batch_size} ({num_iterations} iterations)...")
 
    total_processed = num_iterations * batch_size
    latencies = []
    total_correct = 0  # Track correct preds on the fly to save memory
    
    for i in range(num_iterations):
        batch_start = i * batch_size
        batch_end = batch_start + batch_size
        
        # Wrap around dataset pool if necessary using np.arange
        indices = np.arange(batch_start, batch_end) % dataset_size
        
        # Data is already in numpy format
        batch_input = X_test_batch[indices]
        batch_expected = y_expected_batch[indices]
        
        # Run inference and measure latency using CPU perf_counter
        start_time = time.perf_counter()
        raw_output = session.run(None, {input_name: batch_input})[0]
        end_time = time.perf_counter()
        
        # Convert seconds to milliseconds
        latencies.append((end_time - start_time) * 1000)
        
        # Tally accuracy for this batch against all 3 gates simultaneously
        preds = (raw_output > 0.5).astype(np.int32)
        total_correct += np.all(preds == batch_expected, axis=1).sum()
    
    # 3. Compute statistics
    lat_np = np.array(latencies)
    total_time_sec = np.sum(lat_np) / 1000
    throughput = total_processed / total_time_sec
    total_acc = (total_correct / total_processed) * 100
    
    batch_results.append({
        'batch_size': batch_size,
        'accuracy': total_acc,
        'throughput_ksps': throughput / 1000,
        'min': np.min(lat_np),
        'mean': np.mean(lat_np),
        'median': np.median(lat_np),
        'max': np.max(lat_np),
        'p95': np.percentile(lat_np, 95),
        'p99': np.percentile(lat_np, 99)
    })
 
# 4. Generate Markdown table
table_header = "| Batch Size | Accuracy | Throughput (ksamp/s) | Latency Mean (ms) | Latency Min (ms) | Latency Median (ms) | Latency p95 (ms) | Latency p99 (ms) | Latency Max (ms) | \n"
table_divider = "| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n"
table_rows = ""
 
for r in batch_results:
    table_rows += (f"| {r['batch_size']} | {r['accuracy']:.2f}% | {r['throughput_ksps']:.2f} | "
                   f"{r['mean']:.4f} | {r['min']:.4f} | {r['median']:.4f} | {r['p95']:.4f} | {r['p99']:.4f} | {r['max']:.4f} |\n")
 
markdown_output = f"""
### BENCHMARK RESULTS (20k Iterations/Batch Size)
 
{table_header}{table_divider}{table_rows}
 
**Iterations per Batch Size:** {num_iterations}  
**Device:** {device}  
**Note:** Data is continually wrapped around a {dataset_size}-sample pool.  
**Note:** Latency is measured for each batch, not per sample.  
"""
 
display(Markdown(markdown_output))

>>> Starting benchmark for batch size: 1 (20000 iterations)...
>>> Starting benchmark for batch size: 8 (20000 iterations)...
>>> Starting benchmark for batch size: 64 (20000 iterations)...
>>> Starting benchmark for batch size: 512 (20000 iterations)...
>>> Starting benchmark for batch size: 2048 (20000 iterations)...
>>> Starting benchmark for batch size: 16384 (20000 iterations)...



### BENCHMARK RESULTS (20k Iterations/Batch Size)

| Batch Size | Accuracy | Throughput (ksamp/s) | Latency Mean (ms) | Latency Min (ms) | Latency Median (ms) | Latency p95 (ms) | Latency p99 (ms) | Latency Max (ms) | 
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 100.00% | 148.53 | 0.0067 | 0.0057 | 0.0061 | 0.0088 | 0.0115 | 0.4249 |
| 8 | 100.00% | 1296.80 | 0.0062 | 0.0057 | 0.0059 | 0.0061 | 0.0083 | 0.3923 |
| 64 | 100.00% | 9760.67 | 0.0066 | 0.0060 | 0.0062 | 0.0065 | 0.0092 | 0.4110 |
| 512 | 100.00% | 57773.30 | 0.0089 | 0.0078 | 0.0081 | 0.0091 | 0.0138 | 0.3951 |
| 2048 | 100.00% | 133722.31 | 0.0153 | 0.0142 | 0.0148 | 0.0167 | 0.0289 | 0.3583 |
| 16384 | 100.00% | 240541.15 | 0.0681 | 0.0522 | 0.0581 | 0.0933 | 0.1181 | 0.4540 |


**Iterations per Batch Size:** 20000  
**Device:** GPU  
**Note:** Data is continually wrapped around a 100000-sample pool.  
**Note:** Latency is measured for each batch, not per sample.  
